# Insurgent Primaries, 2018–2026: Money, Outside Spending, and Outcomes

The case universe is every House candidate endorsed by Justice Democrats from 2018 through 2026 (144 endorsement rows from the organization's Wikipedia tables, which carry primary result and vote share), matched to FEC data for the candidate and for their main Democratic-primary opponent — the incumbent if the JD candidate was a challenger, else the best-funded other Democrat in the district. Justice Democrats is used as the universe because it is the one insurgent-left organization with a complete, dated endorsement record across all five cycles; DSA's federal endorsements overlap heavily with it (Ocasio-Cortez, Tlaib, Bush, Bowman, Lee, Casar, Kiros, Valdez, Avila Chevalier, McKinney).

**Questions.** Do JD candidates win when outspent, and how often? Does outside spending against them predict losing? How do the money ratios in the wins (Ocasio-Cortez 2018, Bowman 2020, Bush 2020, Lee 2022, Kiros / McKinney / Avila Chevalier 2026) compare with the losses (Cisneros 2020 and 2022, Turner 2021, Bowman and Bush 2024, Allam and Abughazaleh 2026)?

**Inputs:** `data/raw/insurgent/jd_cases_fec.csv` from `src/data_collection/fec_insurgent_pull.py`. Money is receipts through the last FEC report filed before the primary (year-to-date plus the prior year-end), which is the cleanest pre-primary measure available without itemized-transaction pulls; independent expenditures are Schedule E totals for the cycle by support/oppose.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')
RAW_DIR = Path('../data/raw'); TABLES_DIR = Path('../outputs/tables')
pd.set_option('display.width', 240)
d = pd.read_csv(RAW_DIR / 'insurgent' / 'jd_cases_fec.csv')
# 'Advanced' = made a top-two general (CA/WA) or a runoff; runoff outcomes and blank results are set by hand from the Wikipedia table
OVERRIDE = {(2022, 'Jessica Cisneros'): ('Lost', 49.69), (2022, 'Kina Collins'): ('Lost', 45.72), (2022, 'Marie Newman'): ('Lost', 29.23), (2018, 'Lorie Burch'): ('Won', 75.04)}
for (cy, nm), (res, pct) in OVERRIDE.items():
    d.loc[(d.cycle == cy) & (d.candidate == nm), ['prim_result', 'prim_pct']] = [res, f'{pct}%']
# Ideological tier. 'DSA' = a DSA member or DSA-endorsed (national or chapter) per the public record
# (Ballotpedia's DSA endorsement tables 2018-2026; DSA statements; candidates' Wikipedia articles). Everyone else in the
# universe is a Justice Democrats-only progressive (JD's criteria already require Medicare for All and no corporate PAC money).
DSA = {'Alexandria Ocasio-Cortez', 'Rashida Tlaib', 'Cori Bush', 'Jamaal Bowman', 'Ilhan Omar', 'Summer Lee', 'Greg Casar', 'Melat Kiros', 'Donavan McKinney',
       'Chris Rabb', 'Claire Valdez', 'Darializa Avila Chevalier', 'Kaniela Ing', 'Sarah Smith'}
d['tier'] = np.where(d.candidate.isin(DSA), 'DSA', 'JD-only')
# Top-two 'Advanced' for a non-incumbent (CA/WA) is neither a primary win nor a loss; excluded from win rates
d['pending'] = (d.prim_result == 'Advanced') & (~d.jd_incumbent) & d.state.isin(['CA', 'WA'])
d['won'] = d.prim_result.isin(['Won', 'Advanced']).astype(float)
d['pct'] = pd.to_numeric(d.prim_pct.astype(str).str.rstrip('%'), errors='coerce')
d['contested'] = (d.pct < 99) & (~d.pending)
d['type'] = np.select([d.jd_incumbent, d.opp_is_incumbent], ['JD incumbent', 'challenger vs incumbent'], 'open / other')
for c in ['pre_primary_receipts', 'opp_pre_primary_receipts', 'ie_support', 'ie_oppose', 'opp_ie_support', 'opp_ie_oppose', 'cycle_receipts', 'opp_cycle_receipts',
          'ie_support_pre', 'ie_oppose_pre', 'opp_ie_support_pre', 'opp_ie_oppose_pre']:
    d[c] = pd.to_numeric(d[c], errors='coerce').fillna(0) / 1e6
# Money sides use independent expenditures made on or before the primary date (itemized Schedule E), not cycle totals
d['jd_side'] = d.pre_primary_receipts + d.ie_support_pre + d.opp_ie_oppose_pre
d['opp_side'] = d.opp_pre_primary_receipts + d.opp_ie_support_pre + d.ie_oppose_pre
d['money_ratio'] = d.opp_side / d.jd_side.replace(0, np.nan)
d['outspent'] = d.opp_side > d.jd_side
d['small_share'] = pd.to_numeric(d.pre_primary_unitemized, errors='coerce') / (pd.to_numeric(d.pre_primary_receipts, errors='coerce') * 1e6)
print(len(d), 'cases;', d.contested.sum(), 'contested;', d.pending.sum(), 'pending top-two')
print(d.groupby(['cycle', 'type']).size().unstack(fill_value=0))
print(d.groupby(['tier', 'type']).size().unstack(fill_value=0))

133 cases; 106 contested; 5 pending top-two
type   JD incumbent  challenger vs incumbent  open / other
cycle                                                     
2018              3                        9            54
2020              7                        6             1
2021              0                        0             1
2022             10                        3             3
2024             12                        0             0
2026              9                        7             8
type     JD incumbent  challenger vs incumbent  open / other
tier                                                        
DSA                20                        9             7
JD-only            21                       16            60


## 1. Win rates by case type, cycle, and money position

In [2]:
c = d[d.contested].copy()
print('Contested primaries only (JD candidate < 99%):')
print(c.groupby('type').apply(lambda s: pd.Series({'N': len(s), 'wins': s.won.sum(), 'win_rate': s.won.mean(), 'outspent_share': s.outspent.mean(), 'median_money_ratio': s.money_ratio.median()})).round(2))
print()
print('By ideological tier (DSA member/endorsed vs JD-only):')
print(c.groupby(['tier', 'type']).apply(lambda s: pd.Series({'N': len(s), 'wins': s.won.sum(), 'win_rate': s.won.mean(), 'outspent_share': s.outspent.mean(), 'median_money_ratio': s.money_ratio.median(), 'mean_pct': s.pct.mean()})).round(2))
print()
print(c.groupby('cycle').apply(lambda s: pd.Series({'N': len(s), 'wins': s.won.sum(), 'win_rate': s.won.mean(), 'outspent_share': s.outspent.mean()})).round(2))
print()
print('Win rate by whether the JD side was outspent (contested):')
print(c.groupby(['type', 'outspent']).apply(lambda s: pd.Series({'N': len(s), 'win_rate': s.won.mean(), 'mean_pct': s.pct.mean()})).round(2))
c.groupby(['type', 'outspent']).apply(lambda s: pd.Series({'N': len(s), 'win_rate': s.won.mean()})).to_csv(TABLES_DIR / 'insurgent_winrate_by_money.csv')

Contested primaries only (JD candidate < 99%):
                            N  wins  win_rate  outspent_share  median_money_ratio
type                                                                             
JD incumbent             28.0  25.0      0.89            0.11                0.01
challenger vs incumbent  21.0   8.0      0.38            0.76                2.45
open / other             57.0  19.0      0.33            0.70                1.62

By ideological tier (DSA member/endorsed vs JD-only):
                                    N  wins  win_rate  outspent_share  median_money_ratio  mean_pct
tier    type                                                                                       
DSA     JD incumbent             17.0  15.0      0.88            0.12                0.03     66.20
        challenger vs incumbent   8.0   6.0      0.75            1.00                2.32     48.59
        open / other              7.0   5.0      0.71            0.43                0.7

## 2. The challenger-vs-incumbent cases, one by one

In [3]:
ch = c[c.type == 'challenger vs incumbent'].sort_values(['cycle', 'pct'], ascending=[True, False])
cols = ['cycle', 'candidate', 'tier', 'state', 'district', 'opp_name', 'prim_result', 'pct', 'pre_primary_receipts', 'opp_pre_primary_receipts', 'ie_support_pre', 'ie_oppose_pre', 'opp_ie_support_pre', 'opp_ie_oppose_pre', 'money_ratio', 'small_share', 'ie_top_opposers']
print(ch[cols].round(2).to_string(index=False))
ch[cols].to_csv(TABLES_DIR / 'insurgent_challengers_vs_incumbents.csv', index=False)

 cycle                 candidate    tier state  district                         opp_name prim_result   pct  pre_primary_receipts  opp_pre_primary_receipts  ie_support_pre  ie_oppose_pre  opp_ie_support_pre  opp_ie_oppose_pre  money_ratio  small_share                                                                                     ie_top_opposers
  2018           Ayanna Pressley JD-only    MA         7               CAPUANO, MICHAEL E         Won 58.60                  0.90                      2.08            0.06           0.00                0.28               0.00         2.45         0.22                                                                                                 NaN
  2018  Alexandria Ocasio-Cortez     DSA    NY        14                  CROWLEY, JOSEPH         Won 56.74                  0.37                      5.54            0.00           0.00                0.00               0.01        14.40         0.69                                             

## 3. Outside spending against insurgents: who, how much, and did it work?

In [4]:
d['ie_against_any'] = d.ie_oppose_pre > 0.25
print('Cases with > $250k in independent expenditures against the JD candidate BEFORE the primary:')
big = d[d.ie_against_any].sort_values('ie_oppose_pre', ascending=False)
print(big[['cycle', 'candidate', 'state', 'district', 'type', 'tier', 'prim_result', 'pct', 'ie_oppose_pre', 'ie_oppose', 'ie_support_pre', 'jd_side', 'opp_side', 'ie_top_opposers']].round(2).to_string(index=False))
print('\nWin rate: with > $250k against %.2f (N=%d); without %.2f (N=%d) [contested only]' % (
    c[c.ie_oppose_pre > 0.25].won.mean(), (c.ie_oppose_pre > 0.25).sum(), c[c.ie_oppose_pre <= 0.25].won.mean(), (c.ie_oppose_pre <= 0.25).sum()))
# who spends against: aggregate top opposers text
import re
from collections import Counter
cnt = Counter()
for s in d.ie_top_opposers.dropna():
    for part in s.split(';'):
        m = re.match(r'\s*(.+?)\s+([\d.]+)M', part)
        if m: cnt[m.group(1).strip()] += float(m.group(2))
print('\nLargest opposing committees across all cases ($M, top-3 per case summed):')
for k, v in cnt.most_common(12): print(f'  {k[:50]:50s} {v:6.2f}')

Cases with > $250k in independent expenditures against the JD candidate BEFORE the primary:
 cycle                 candidate state  district                    type    tier prim_result   pct  ie_oppose_pre  ie_oppose  ie_support_pre  jd_side  opp_side                                                                                                          ie_top_opposers
  2024             Jamaal Bowman    NY        16            JD incumbent     DSA        Lost 41.41          12.04      12.04            1.95     8.88     24.84                                                  UNITED DEMOCRACY PROJECT ('UDP') 9.87M; FAIRSHAKE 2.08M; DMFI PAC 0.10M
  2024                 Cori Bush    MO         1            JD incumbent     DSA        Lost 45.64           7.34       7.34            2.38     7.46     17.48                                  UNITED DEMOCRACY PROJECT ('UDP') 5.24M; FAIRSHAKE 1.41M; MAINSTREAM DEMOCRATS PAC 0.60M
  2026                 Cori Bush    MO         1 challenger vs in

## 4. Models: money ratio, outside spending, incumbency, and the vote share

In [5]:
m = c.dropna(subset=['money_ratio']).copy()
m['log_ratio'] = np.log(m.money_ratio.clip(0.05, 50))
m['log_ie_oppose'] = np.log1p(m.ie_oppose_pre)
m['dsa'] = (m.tier == 'DSA').astype(float)
m['vs_inc'] = (m.type == 'challenger vs incumbent').astype(float); m['jd_inc'] = (m.type == 'JD incumbent').astype(float)
m['post2020'] = (m.cycle >= 2022).astype(float)
X = sm.add_constant(m[['log_ratio', 'log_ie_oppose', 'vs_inc', 'jd_inc', 'post2020', 'dsa']])
r = sm.OLS(m.pct, X).fit(cov_type='HC1'); print('Primary vote share (%%) on money and type, contested cases (N=%d):' % len(m)); print(pd.DataFrame({'coef': r.params, 'se': r.bse, 'p': r.pvalues}).round(2))
r2 = sm.Logit(m.won, X).fit(disp=0, cov_type='HC1'); print('\nWin (logit):'); print(pd.DataFrame({'coef': r2.params, 'se': r2.bse, 'p': r2.pvalues}).round(2))
pd.DataFrame({'ols_coef': r.params, 'ols_se': r.bse, 'logit_coef': r2.params, 'logit_se': r2.bse}).to_csv(TABLES_DIR / 'insurgent_models.csv')

Primary vote share (%) on money and type, contested cases (N=106):
                coef    se     p
const          35.43  2.44  0.00
log_ratio      -7.87  0.99  0.00
log_ie_oppose  -2.91  2.06  0.16
vs_inc         12.84  2.83  0.00
jd_inc         11.06  4.20  0.01
post2020        0.31  2.77  0.91
dsa             6.47  3.26  0.05

Win (logit):
               coef    se     p
const         -0.59  0.43  0.16
log_ratio     -0.85  0.27  0.00
log_ie_oppose -2.15  0.66  0.00
vs_inc        -0.01  0.74  0.99
jd_inc         1.19  0.75  0.11
post2020       0.26  0.59  0.66
dsa            2.98  1.21  0.01


## 5. Small-dollar share

In [6]:
print('Small-dollar (unitemized) share of pre-primary receipts, JD candidates vs opponents:')
d['opp_small_share'] = pd.to_numeric(d.opp_pre_primary_unitemized, errors='coerce') / (pd.to_numeric(d.opp_pre_primary_receipts, errors='coerce') * 1e6)
print(d.groupby('type')[['small_share', 'opp_small_share']].median().round(2))
print(d.groupby('cycle')[['small_share', 'opp_small_share']].median().round(2))
print('\nWinners vs losers among contested JD candidates (median):'); print(c.groupby('won')[['small_share', 'pre_primary_receipts', 'money_ratio', 'ie_oppose_pre']].median().round(2))
print('\nBy tier (median, contested):'); print(c.groupby('tier')[['small_share', 'pre_primary_receipts', 'money_ratio', 'ie_oppose_pre', 'pct', 'won']].median().round(2))

Small-dollar (unitemized) share of pre-primary receipts, JD candidates vs opponents:
                         small_share  opp_small_share
type                                                 
JD incumbent                    0.28             0.05
challenger vs incumbent         0.39             0.02
open / other                    0.35             0.12
       small_share  opp_small_share
cycle                              
2018          0.35             0.12
2020          0.41             0.01
2021          0.56             0.04
2022          0.32             0.05
2024          0.24             0.07
2026          0.32             0.03

Winners vs losers among contested JD candidates (median):
     small_share  pre_primary_receipts  money_ratio  ie_oppose_pre
won                                                               
0.0         0.35                  0.50         2.73            0.0
1.0         0.34                  2.53         0.20            0.0

By tier (median, contested):


## 6. Media coverage of the challenger cases

GDELT coverage volume and tone for each challenger and the incumbent they ran against, over the 90 days ending on the primary date. The window stops at the primary deliberately: coverage after a result is largely a consequence of it (AOC's volume multiplied roughly a hundredfold the week she won), so it cannot go into a variable about the result.

Coverage turned out to be far deeper than expected for House primaries — 41 of 42 candidate-series have at least 20 days with coverage, the exception being Sameena Mustafa (IL-5 2018) at three. Median covered days are 45 for challengers and 64 for incumbents.

Two limits carry through everything below. Queries are name phrases with no district context, so common names pick up unrelated coverage — a particular risk for the incumbents Danny Davis, Mike Quigley and Wesley Bell, and for the challenger Marie Newman, whose 2018 volume exceeds Lipinski's by eight to one in a race she lost by two points. And GDELT tone is a lexicon score on article text: as Part 10 documents at length, it measures word valence, not framing, and cannot tell an attack from a report of one.

In [7]:
from scipy import stats
GF = RAW_DIR / 'gdelt_insurgent' / 'gdelt_insurgent_daily.csv'
if not GF.exists():
    print('insurgent GDELT file not on disk; run src/data_collection/gdelt_insurgent_pull.py')
else:
    gi = pd.read_csv(GF)
    def wtone(x):
        ok = (x.volume > 0) & x.tone.notna()
        return np.average(x.tone[ok], weights=x.volume[ok]) if ok.any() else np.nan
    si = gi.groupby(['case_id','role','name','tier','won']).apply(lambda x: pd.Series({
        'covered': int((x.volume > 0).sum()), 'vol': x.volume.mean()*1000, 'tone': wtone(x)})).reset_index()
    print(f'candidate-series: {len(si)}; with >=10 covered days: {(si.covered>=10).sum()}')
    si = si[si.covered >= 10]
    chs, ins = si[si.role=='challenger'], si[si.role=='incumbent']

    print('\n1. CHALLENGER COVERAGE BY TIER')
    rows=[]
    for tier in ['DSA','JD-only']:
        t = chs[chs.tier==tier]
        rows.append({'tier':tier,'n':len(t),'tone':t.tone.mean(),'tone_sd':t.tone.std(),'vol_mean':t.vol.mean(),'vol_median':t.vol.median()})
    tt = pd.DataFrame(rows); print(tt.round(2).to_string(index=False))
    A,B = chs[chs.tier=='DSA'], chs[chs.tier=='JD-only']
    _,p_t = stats.ttest_ind(A.tone,B.tone,equal_var=False); _,p_v = stats.ttest_ind(A.vol,B.vol,equal_var=False)
    print(f"   DSA - JD-only: tone {A.tone.mean()-B.tone.mean():+.2f} (p={p_t:.3f}), volume {A.vol.mean()-B.vol.mean():+.2f} (p={p_v:.3f}) -- null on both")
    tt.to_csv(TABLES_DIR / 'insurgent_media_by_tier.csv', index=False)

    print('\n2. CHALLENGER vs THE INCUMBENT THEY FACED (paired within case)')
    J = chs.merge(ins, on='case_id', suffixes=('_ch','_in'))
    J['tone_gap'] = J.tone_ch - J.tone_in; J['vol_ratio'] = J.vol_ch / J.vol_in
    t_, p_ = stats.ttest_1samp(J.tone_gap.dropna(), 0)
    print(f"   tone gap mean {J.tone_gap.mean():+.2f}, median {J.tone_gap.median():+.2f}; paired t={t_:+.2f} p={p_:.3f}")
    print(f"   {(J.tone_gap<0).sum()} of {len(J)} challengers were covered MORE negatively than their incumbent")
    print(f"   volume ratio challenger/incumbent: median {J.vol_ratio.median():.2f}")
    for tier in ['DSA','JD-only']:
        t = J[J.tier_ch==tier]
        print(f"     {tier:8s} tone gap {t.tone_gap.mean():+.2f}, volume ratio median {t.vol_ratio.median():.2f}")
    J.to_csv(TABLES_DIR / 'insurgent_media_paired.csv', index=False)

    print('\n3. BY OUTCOME')
    for w,lab in [(1,'won'),(0,'lost')]:
        t = chs[chs.won==w]; print(f"   {lab:5s} n={len(t):2d}  tone {t.tone.mean():+.2f}  volume {t.vol.mean():.2f}")
    W,L = chs[chs.won==1], chs[chs.won==0]
    _,p1 = stats.ttest_ind(W.tone,L.tone,equal_var=False); _,p2 = stats.ttest_ind(W.vol,L.vol,equal_var=False)
    print(f"   winners - losers: tone {W.tone.mean()-L.tone.mean():+.2f} (p={p1:.3f}), volume {W.vol.mean()-L.vol.mean():+.2f} (p={p2:.3f})")
    print("\n   The outcome difference is null before any control is applied, so no model of outcome on")
    print("   tone net of money is fitted: with 20 cases and 8 wins it would be fitting noise to a")
    print("   descriptive null, and pre-primary coverage partly reflects expected viability anyway.")

    print('\n4. PER CASE')
    print(J[['case_id','tier_ch','won_ch','name_ch','tone_ch','tone_in','tone_gap','vol_ch','vol_in']]
          .round(2).sort_values('tone_gap').to_string(index=False))

candidate-series: 42; with >=10 covered days: 41

1. CHALLENGER COVERAGE BY TIER
   tier  n  tone  tone_sd  vol_mean  vol_median
    DSA  8 -1.10     0.87      1.69        1.18
JD-only 12 -0.89     0.55      1.95        1.42
   DSA - JD-only: tone -0.21 (p=0.558), volume -0.27 (p=0.660) -- null on both

2. CHALLENGER vs THE INCUMBENT THEY FACED (paired within case)
   tone gap mean +0.70, median +0.42; paired t=+3.71 p=0.001
   5 of 20 challengers were covered MORE negatively than their incumbent
   volume ratio challenger/incumbent: median 0.69
     DSA      tone gap +1.16, volume ratio median 0.64
     JD-only  tone gap +0.39, volume ratio median 0.71

3. BY OUTCOME
   won   n= 8  tone -1.08  volume 2.13
   lost  n=12  tone -0.90  volume 1.66
   winners - losers: tone -0.18 (p=0.625), volume +0.47 (p=0.500)

   The outcome difference is null before any control is applied, so no model of outcome on
   tone net of money is fitted: with 20 cases and 8 wins it would be fitting noise to a

## Limitations

- **Universe = Justice Democrats endorsements.** Insurgent-left candidates JD did not endorse (e.g., Brad Lander 2026, Shahid Buttar 2020) are excluded; JD's 2018 slate included many open-seat and general-election-only candidates, which is why the analysis conditions on contested primaries and separates the challenger-vs-incumbent cases.
- **Opponent identification is mechanical:** the Democratic incumbent, else the best-funded other Democrat by cycle receipts. In multi-candidate open seats the "opponent" is one of several.
- **Pre-primary receipts are year-to-date at the last report before the primary plus the prior year-end;** where the FEC has no 12P report the prior quarterly is used. Independent expenditures are cycle totals, which for incumbents include general-election spending; for primary-only losers they are effectively primary totals.
- **2026 outcomes are as recorded on Wikipedia at pull time**; 2026 FEC totals are partial (cycle still open).
- **The DSA tier is hand-coded** from Ballotpedia's DSA endorsement tables, DSA statements, and candidates' articles; chapter endorsements count. Fourteen of 133 cases.
- **Small N.** Roughly 25 challenger-vs-incumbent cases across five cycles; the models are descriptive.